# nb02 — SQL + pandas Parity: Reproduce Every Tableau Number

<hr style="border: 3px solid black;">

**Purpose:** prove that every number in the Tableau workbook is reproducible in SQL (DuckDB) and pandas — the "same numbers, three ways" evidence for the blog post.

Each section = one Tableau sheet: the business question, the SQL, the pandas, and a comparison against the value verified in Tableau on July 8, 2026.

**Requires:** `pip install duckdb` (one time). Data: `../data/medi_cal_enrollment_clean.csv`.

In [1]:
from pathlib import Path

import duckdb
import pandas as pd

DATA = Path('..') / 'data' / 'medi_cal_enrollment_clean.csv'
OUT = Path('..') / 'data' / 'parity'
OUT.mkdir(exist_ok=True)

df = pd.read_csv(DATA, parse_dates=['Month'])
con = duckdb.connect()
con.execute("CREATE OR REPLACE VIEW enr AS SELECT * FROM read_csv_auto('" + str(DATA) + "')")
print(df.shape)  # expect (30431, 8)

(30431, 8)


## 1. Sheet `Avg Enrollment Right vs Wrong`

**Question:** what is LA County's average monthly Medical enrollment?

**Tableau verified:** Correct = 2,546,791 · Incorrect = 437,233

In [2]:
sql = con.execute('''
    SELECT AVG(Enrollees)                                   AS incorrect_avg,
           SUM(Enrollees) / COUNT(DISTINCT Month)           AS correct_avg
    FROM enr
    WHERE County = 'Los Angeles' AND "Plan Category" = 'Medical'
''').df()

la = df[(df.County == 'Los Angeles') & (df['Plan Category'] == 'Medical')]
pds = pd.DataFrame([{
    'incorrect_avg': la.Enrollees.mean(),
    'correct_avg': la.Enrollees.sum() / la.Month.nunique(),
}])

parity = pd.concat([sql.assign(engine='SQL'), pds.assign(engine='pandas')])
parity['tableau_correct'] = 2546791
parity['tableau_incorrect'] = 437233
parity.round(0).to_csv(OUT / 'parity_01_avg_right_wrong.csv', index=False)
parity.round(0)

,incorrect_avg,correct_avg,engine,tableau_correct,tableau_incorrect
0,437233.0,2546791.0,SQL,2546791,437233
0,437233.0,2546791.0,pandas,2546791,437233


## 2. Sheet `FIXED % of County`

**Question:** what share of LA County's Medical enrollment does each plan hold in June 2026?

**Tableau verified:** L.A. Care 59.4% · Health Net 30.5% · Kaiser 9.6% (sums to 100%)

The FIXED LOD ↔ window function ↔ groupby transform. The two context filters (Month, Plan Category) become the WHERE clause — SQL runs WHERE before window functions, which is exactly what Add to Context did in Tableau.

In [3]:
sql = con.execute('''
    WITH la AS (
        SELECT "Plan" AS plan, SUM(Enrollees) AS members,
               SUM(SUM(Enrollees)) OVER ()      AS county_total   -- the FIXED denominator
        FROM enr
        WHERE County = 'Los Angeles' AND "Plan Category" = 'Medical'
          AND Month = DATE '2026-06-01'
        GROUP BY 1
    )
    SELECT plan, members, ROUND(100.0 * members / county_total, 1) AS pct_of_county
    FROM la ORDER BY members DESC
''').df()

la6 = df[(df.County == 'Los Angeles') & (df['Plan Category'] == 'Medical') & (df.Month == '2026-06-01')]
p = la6.groupby('Plan', as_index=False)['Enrollees'].sum()
p['pct_of_county'] = (100 * p.Enrollees / p.Enrollees.sum()).round(1)
p = p.sort_values('Enrollees', ascending=False)

sql.to_csv(OUT / 'parity_02_fixed_pct_county.csv', index=False)
print(sql.head(3))
print(p.head(3))  # both: L.A. Care 59.4, Health Net 30.5, Kaiser 9.6

                             plan    members  pct_of_county
0           L.A. Care Health Plan  2102545.0           59.4
1  Health Net Community Solutions  1079978.0           30.5
2               Kaiser Permanente   339151.0            9.6
                             Plan  Enrollees  pct_of_county
5           L.A. Care Health Plan  2102545.0           59.4
3  Health Net Community Solutions  1079978.0           30.5
4               Kaiser Permanente   339151.0            9.6


## 3. Sheet `INCLUDE Avg per Plan`

**Question:** what is the typical plan size within each plan type, June 2026 statewide?

**Tableau verified:** Local Initiative (2 Plan) 652,110 vs naive 434,740

INCLUDE = aggregate finer first (per plan), then average up — a two step groupby.

In [4]:
sql = con.execute('''
    WITH per_plan AS (
        SELECT "Plan Type" AS plan_type, "Plan" AS plan, SUM(Enrollees) AS plan_total
        FROM enr
        WHERE "Plan Category" = 'Medical' AND Month = DATE '2026-06-01'
        GROUP BY 1, 2
    ),
    naive AS (
        SELECT "Plan Type" AS plan_type, AVG(Enrollees) AS naive_avg
        FROM enr
        WHERE "Plan Category" = 'Medical' AND Month = DATE '2026-06-01'
        GROUP BY 1
    )
    SELECT p.plan_type, ROUND(AVG(p.plan_total)) AS include_avg, ROUND(ANY_VALUE(n.naive_avg)) AS naive_avg
    FROM per_plan p JOIN naive n USING (plan_type)
    GROUP BY 1 ORDER BY include_avg DESC
''').df()

m6 = df[(df['Plan Category'] == 'Medical') & (df.Month == '2026-06-01')]
include = m6.groupby(['Plan Type', 'Plan'])['Enrollees'].sum().groupby('Plan Type').mean().round()
naive = m6.groupby('Plan Type')['Enrollees'].mean().round()
pds = pd.DataFrame({'include_avg': include, 'naive_avg': naive}).sort_values('include_avg', ascending=False)

sql.to_csv(OUT / 'parity_03_include_avg.csv', index=False)
print(sql.head(3))
print(pds.head(3))  # Local Initiative (2 Plan): 652110 vs 434740

                         plan_type  include_avg  naive_avg
0        Local Initiative (2 Plan)     652110.0   434740.0
1         Commercial Plan (2 Plan)     421397.0    87186.0
2  County Organized Health Systems     316089.0    61844.0
                                 include_avg  naive_avg
Plan Type                                              
Local Initiative (2 Plan)           652110.0   434740.0
Commercial Plan (2 Plan)            421397.0    87186.0
County Organized Health Systems     316089.0    61844.0


## 4. Sheet `EXCLUDE Benchmark`

**Question:** what share of statewide Medical enrollment does each county contribute, June 2026?

**Tableau verified:** Los Angeles 27.5%

EXCLUDE = a window total over fewer columns than the view; note the Tableau version needed MIN() around the LOD — SQL and pandas have no such wrinkle because the window value is used directly.

In [5]:
sql = con.execute('''
    SELECT County,
           ROUND(100.0 * SUM(Enrollees) / SUM(SUM(Enrollees)) OVER (), 1) AS pct_of_state
    FROM enr
    WHERE "Plan Category" = 'Medical' AND Month = DATE '2026-06-01'
    GROUP BY 1 ORDER BY pct_of_state DESC
''').df()

c = m6.groupby('County', as_index=False)['Enrollees'].sum()
c['pct_of_state'] = (100 * c.Enrollees / c.Enrollees.sum()).round(1)
c = c.sort_values('pct_of_state', ascending=False)

sql.to_csv(OUT / 'parity_04_exclude_benchmark.csv', index=False)
print(sql.head(3))
print(c.head(3))  # Los Angeles 27.5

           County  pct_of_state
0     Los Angeles          27.5
1  San Bernardino           6.8
2       Riverside           6.7
            County  Enrollees  pct_of_state
18     Los Angeles  3542474.0          27.5
35  San Bernardino   870062.0           6.8
32       Riverside   867204.0           6.7


## 5. Sheet `Running Total + YoY`

**Question:** statewide cumulative growth and year over year change.

Table calcs = window functions over the query result: running total = `SUM() OVER (ORDER BY ...)`, YoY = `LAG(..., 12)`. Checked at June 2026 against the workbook tooltip.

In [6]:
sql = con.execute('''
    WITH monthly AS (
        SELECT Month, SUM(Enrollees) AS members
        FROM enr WHERE "Plan Category" = 'Medical'
        GROUP BY 1
    )
    SELECT Month, members,
           SUM(members) OVER (ORDER BY Month)                              AS running_total,
           ROUND(100.0 * (members - LAG(members, 12) OVER (ORDER BY Month))
                 / LAG(members, 12) OVER (ORDER BY Month), 1)              AS yoy_pct
    FROM monthly ORDER BY Month
''').df()

mo = df[df['Plan Category'] == 'Medical'].groupby('Month', as_index=False)['Enrollees'].sum()
mo['running_total'] = mo.Enrollees.cumsum()
mo['yoy_pct'] = (100 * mo.Enrollees.pct_change(12)).round(1)

sql.to_csv(OUT / 'parity_05_running_yoy.csv', index=False)
print(sql.tail(3))
print(mo.tail(3))

         Month     members  running_total  yoy_pct
231 2026-04-01  13218640.0   2.045607e+09     -5.7
232 2026-05-01  13066144.0   2.058673e+09     -6.8
233 2026-06-01  12888522.0   2.071562e+09     -8.2
         Month   Enrollees  running_total  yoy_pct
231 2026-04-01  13218640.0   2.045607e+09     -5.7
232 2026-05-01  13066144.0   2.058673e+09     -6.8
233 2026-06-01  12888522.0   2.071562e+09     -8.2


## 6. Sheet `Top N + Metric Swap` (as redesigned: top plans WITHIN the county)

**Question:** the top 10 plans in a chosen county, June 2026.

**Tableau verified (LA County):** 7 plans, L.A. Care 2,102,545 on top.

The parameter becomes a plain variable; the context filter fix becomes filter first, rank second.

In [7]:
TOP_N = 10
COUNTY = 'Los Angeles'

sql = con.execute(f'''
    SELECT "Plan" AS plan, SUM(Enrollees) AS members
    FROM enr
    WHERE "Plan Category" = 'Medical' AND Month = DATE '2026-06-01'
      AND County = '{COUNTY}'          -- filter FIRST (the Add to Context fix)
    GROUP BY 1 ORDER BY members DESC LIMIT {TOP_N}
''').df()

topn = (m6[m6.County == COUNTY].groupby('Plan', as_index=False)['Enrollees'].sum()
        .sort_values('Enrollees', ascending=False).head(TOP_N))

sql.to_csv(OUT / 'parity_06_top_n.csv', index=False)
print(len(sql), 'plans')  # expect 7 for LA
print(sql.head(3))
print(topn.head(3))

7 plans
                             plan    members
0           L.A. Care Health Plan  2102545.0
1  Health Net Community Solutions  1079978.0
2               Kaiser Permanente   339151.0
                             Plan  Enrollees
5           L.A. Care Health Plan  2102545.0
3  Health Net Community Solutions  1079978.0
4               Kaiser Permanente   339151.0


---

**Wrap up:** all six parity tables are in `../data/parity/`. Compare each printout against the Tableau verified values noted per section — every pair (SQL row vs pandas row) should match to the digit, and both should match Tableau.

**Next:** Phase 5 — the tabbed blog page consumes these tables plus the Tableau Public embed.